# Trích xuất aspect và phân loại polarity — bản tối ưu

Embedding ký tự + từ, BiLSTM–CRF và bộ phân loại polarity theo span. Tất cả trọng số và vocabulary được học từ train, không dùng pretrained.

Nguồn: [SetFit SemEval Restaurants](https://huggingface.co/datasets/tomaarsen/setfit-absa-semeval-restaurants), revision `8885372fe73256f96bb60f65b550538ac5c26047`. Bảng người dùng: Sequence Labeling → ABSA Extraction; primary **aspect span micro-F1 ↑**, bổ sung polarity macro-F1 ↑. Chuỗi đầu vào là tiếng Anh.

Nhóm mọi hàng cùng `text`; `ordinal` là lần xuất hiện tính từ 0. Dữ liệu thật không chồng lấn, có ba aspect train kết thúc giữa từ, nên gán BIO theo **ký tự gốc**. Toàn bộ polarity test rỗng: chỉ đo extraction trên test, polarity trên validation. Nguồn chỉ có các câu chứa aspect; kết quả không đại diện toàn bộ test SemEval gốc.

Notebook chạy độc lập bằng **Restart Kernel → Run All**. Cần mạng để tải dataset lần đầu; chế độ Parquet cục bộ có kiểm tra SHA-256. Notebook này không đọc notebook kia. Chưa huấn luyện đầy đủ và không kèm điểm mô hình đã đo.

**Cách viết training/checkpoint:** training loop được viết trực tiếp trong cell (sklearn dùng fit); lưu gọn khi validation cải thiện. Checkpoint phục vụ inference, không lưu optimizer/scheduler/RNG để resume.


## 1. Chuẩn bị thư viện
Dùng PyTorch của kernel hiện tại; nếu chỉ có bản CPU thì notebook chạy CPU. Để huấn luyện theo thiết kế, chọn runtime CUDA phù hợp với GPU 32 GB. Không tự thay wheel hoặc driver.


In [ ]:
import importlib.util
import subprocess
import sys

# Giữ PyTorch đang có trong kernel; chỉ cài thư viện đọc dữ liệu nếu thiếu.
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("Kernel chưa có PyTorch. Cài bản phù hợp CPU/CUDA rồi chạy lại notebook.")
if importlib.util.find_spec("datasets") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets==4.8.5"])

from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import math
import random
import re
import numpy as np
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
import datasets
from datasets import load_dataset


## 2. Cấu hình cố định
Các hằng số có thể sửa thủ công trước khi chạy. Không cắt câu, không thay batch theo phần cứng, dùng độ chính xác mặc định của PyTorch. Mỗi biến thể ghi cache/checkpoint riêng dưới `artifacts/` tính từ thư mục làm việc của kernel.


In [ ]:
SEED = 42
DATASET_ID = "tomaarsen/setfit-absa-semeval-restaurants"
DATASET_REVISION = "8885372fe73256f96bb60f65b550538ac5c26047"
PRETRAINING_POLICY = "scratch"
PRETRAINED_COMPONENTS = []
VARIANT = "optimized"
USE_LOCAL_DATA = False
LOCAL_DATA_DIR = Path("data/setfit_absa_restaurants")
ARTIFACT_DIR = Path("artifacts") / VARIANT
CACHE_DIR = ARTIFACT_DIR / "dataset_cache"
CHECKPOINT_PATH = ARTIFACT_DIR / "best_model.pt"
VAL_FRACTION = 0.20
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 0.002
WEIGHT_DECAY = 0.0001
GRAD_CLIP = 1.0
POLARITY_LOSS_WEIGHT = 0.25
CHAR_EMBED_DIM = 48
WORD_EMBED_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.3
WORD_MIN_FREQ = 1
LABELS = ("negative", "neutral", "positive", "conflict")
LABEL_TO_ID = {label: index for index, label in enumerate(LABELS)}
TAGS = ("O", "B", "I")
PAD_ID, UNK_ID = 0, 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Thiết bị: {DEVICE}; biến thể: {VARIANT}; batch: {BATCH_SIZE}; tối đa {EPOCHS} epoch")
print(f"Phiên bản: PyTorch {torch.__version__}; datasets {datasets.__version__}")


## 3. Tải dữ liệu, nhóm câu và chia validation
Gộp hàng trùng cùng offset/nhãn; dừng nếu nhãn mâu thuẫn hoặc span chồng lấn. Loại câu train trùng test trước khi chia validation 20% theo seed 42. Giữ nguyên test. Fingerprint chứa toàn bộ nội dung và annotation để so sánh hai notebook.


In [ ]:
def load_raw_data():
    try:
        if USE_LOCAL_DATA:
            paths = {split: LOCAL_DATA_DIR / f"{split}.parquet" for split in ("train", "test")}
            expected = {
                "train": "d82e7c7b8b78bb7280ccfee2ce36f5de04d9fd3d82c8259d852f7d031d5840ac",
                "test": "c91a8ebd0cdbec6999bd6cff844a1081efbc9039afc950a10dc1dc8cbf70083e",
            }
            for split, path in paths.items():
                if not path.is_file():
                    raise FileNotFoundError(f"Thiếu {path}; tải đúng Parquet ở revision đã ghi.")
                if hashlib.sha256(path.read_bytes()).hexdigest() != expected[split]:
                    raise ValueError(f"{path} không khớp SHA-256 của revision đã kiểm tra.")
            return load_dataset("parquet", data_files={k: str(v) for k, v in paths.items()},
                                cache_dir=str(CACHE_DIR))
        return load_dataset(DATASET_ID, revision=DATASET_REVISION, cache_dir=str(CACHE_DIR))
    except Exception as exc:
        raise RuntimeError(
            "Không đọc được bộ dữ liệu yêu cầu. Kiểm tra mạng/quyền Hugging Face, hoặc đặt "
            "USE_LOCAL_DATA=True và cung cấp train.parquet, test.parquet đúng revision tại "
            f"{LOCAL_DATA_DIR}. Không thay bằng bộ dữ liệu khác. Chi tiết: {exc}"
        ) from exc


def group_rows(rows, split):
    grouped = {}
    duplicates = 0
    for row in rows:
        if not {"text", "span", "label", "ordinal"}.issubset(row):
            raise ValueError("Schema phải có text, span, label, ordinal.")
        text, term, ordinal = row["text"], row["span"], row["ordinal"]
        if not isinstance(text, str) or not isinstance(term, str) or not text or not term:
            raise ValueError(f"Văn bản/aspect rỗng hoặc sai kiểu ở {split}: {row}")
        if not isinstance(ordinal, int) or ordinal < 0:
            raise ValueError(f"ordinal phải là số nguyên không âm: {row}")
        matches = list(re.finditer(re.escape(term), text))
        if ordinal >= len(matches):
            raise ValueError(f"Không tìm được lần xuất hiện ordinal={ordinal}: {row}")
        start, end = matches[ordinal].span()  # Chính xác, phân biệt hoa/thường; end loại trừ.
        label = row["label"]
        if label not in LABEL_TO_ID and not (split == "test" and label == ""):
            raise ValueError(f"Polarity không hợp lệ ở {split}: {label!r}")
        label = label if label else None
        record = grouped.setdefault(text, {"sid": hashlib.sha256(text.encode()).hexdigest(),
                                           "text": text, "spans": {}})
        key = (start, end)
        if key in record["spans"]:
            if record["spans"][key] != label:
                raise ValueError(f"Nhãn mâu thuẫn cho cùng vị trí: {row}")
            duplicates += 1
        record["spans"][key] = label
    records = []
    for text in sorted(grouped):
        record = grouped[text]
        record["spans"] = [(a, b, p) for (a, b), p in sorted(record["spans"].items())]
        if any(left[1] > right[0] for left, right in zip(record["spans"], record["spans"][1:])):
            raise ValueError("Có span chồng lấn; cần mô hình span thay cho BIO một lớp.")
        records.append(record)
    print(f"{split}: {len(rows)} hàng → {len(records)} nhóm câu; gộp {duplicates} hàng lặp.")
    return records


raw = load_raw_data()
if set(raw) != {"train", "test"}:
    raise ValueError(f"Split khác revision đã kiểm tra: {list(raw)}")
all_train_records = group_rows(raw["train"], "train")
test_records = group_rows(raw["test"], "test")
# Chỉ dùng danh tính câu test để kiểm tra trùng; không dùng nhãn test để học/chọn mô hình.
test_texts = {record["text"] for record in test_records}
eligible_records = [record for record in all_train_records if record["text"] not in test_texts]
removed_overlap = len(all_train_records) - len(eligible_records)
indices = list(range(len(eligible_records)))
random.Random(SEED).shuffle(indices)
val_size = max(1, math.ceil(VAL_FRACTION * len(indices)))
val_indices = set(indices[:val_size])
train_records = [r for i, r in enumerate(eligible_records) if i not in val_indices]
val_records = [r for i, r in enumerate(eligible_records) if i in val_indices]
if not train_records or not val_records:
    raise ValueError("Không đủ nhóm câu để chia train/validation.")
assert not ({r["sid"] for r in train_records} & {r["sid"] for r in val_records})
assert not ({r["sid"] for r in eligible_records} & {r["sid"] for r in test_records})

def fingerprint(records):
    payload = json.dumps(records, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest()

SPLIT_FINGERPRINTS = {name: fingerprint(records) for name, records in
                      (("train", train_records), ("validation", val_records), ("test", test_records))}
print(f"Loại khỏi train gốc {removed_overlap} nhóm câu trùng test.")
for name, records in (("train", train_records), ("validation", val_records), ("test", test_records)):
    counts = Counter(p if p is not None else "chưa có nhãn" for r in records for _, _, p in r["spans"])
    print(f"{name}: {len(records)} câu; polarity={dict(counts)}; SHA-256={SPLIT_FINGERPRINTS[name]}")
print("Ví dụ train:", train_records[0])
print("Lưu ý: nguồn SetFit chỉ giữ câu có aspect; không phải toàn bộ câu của SemEval gốc.")


## 4. Vocabulary, nhãn BIO, Dataset và padding
Fit vocabulary sau khi chia train. Nhãn O=0, B=1, I=2; PAD ký tự là 0 nhưng padding luôn bị loại bằng mask. Khoảng trắng nằm trong aspect vẫn được gán I; giữ nguyên hoa/thường và offset của câu. Từ viết thường chỉ dùng làm đặc trưng bổ sung ở bản tối ưu.


In [ ]:
# Mỗi ký tự gốc là một bước thời gian: không chuẩn hóa khoảng trắng, không cắt câu.
# Từ chỉ là đặc trưng bổ sung; các nhãn BIO luôn theo vị trí ký tự gốc.
WORD_PATTERN = r"\w+|[^\w\s]"

def words_with_offsets(text):
    return list(re.finditer(WORD_PATTERN, text))

char_counts = Counter(ch for record in train_records for ch in record["text"])
word_counts = Counter(m.group().lower() for record in train_records
                      for m in words_with_offsets(record["text"]))
char_vocab = {"<PAD>": PAD_ID, "<UNK>": UNK_ID}
char_vocab.update({ch: i + 2 for i, ch in enumerate(sorted(char_counts))})
word_vocab = {"<PAD>": PAD_ID, "<UNK>": UNK_ID}
word_vocab.update({word: i + 2 for i, word in enumerate(sorted(
    word for word, count in word_counts.items() if count >= WORD_MIN_FREQ))})

def encode_record(record):
    text = record["text"]
    char_ids = [char_vocab.get(ch, UNK_ID) for ch in text]
    # Khoảng trắng có ID UNK riêng về vai trò; mã ký tự vẫn giữ nguyên loại khoảng trắng.
    word_ids = [UNK_ID] * len(text)
    for match in words_with_offsets(text):
        wid = word_vocab.get(match.group().lower(), UNK_ID)
        word_ids[match.start():match.end()] = [wid] * (match.end() - match.start())
    tags = [0] * len(text)
    for start, end, _ in record["spans"]:
        assert 0 <= start < end <= len(text)
        assert all(tag == 0 for tag in tags[start:end])
        tags[start] = 1
        tags[start + 1:end] = [2] * (end - start - 1)
    return {"char_ids": char_ids, "word_ids": word_ids, "tags": tags, "record": record}


class AspectDataset(Dataset):
    def __init__(self, records):
        self.items = [encode_record(record) for record in records]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]


def collate_batch(items):
    lengths = torch.tensor([len(item["char_ids"]) for item in items], dtype=torch.long)
    width = int(lengths.max())
    batch = {name: torch.zeros(len(items), width, dtype=torch.long)
             for name in ("char_ids", "word_ids", "tags")}
    spans = []
    for i, item in enumerate(items):
        length = len(item["char_ids"])
        for name in batch:
            batch[name][i, :length] = torch.tensor(item[name])
        for start, end, polarity in item["record"]["spans"]:
            if polarity is not None:
                spans.append((i, start, end, LABEL_TO_ID[polarity]))
    batch["lengths"] = lengths  # pack_padded_sequence nhận lengths trên CPU.
    batch["mask"] = torch.arange(width)[None, :] < lengths[:, None]
    batch["gold_spans"] = torch.tensor(spans, dtype=torch.long).reshape(-1, 4)
    batch["records"] = [item["record"] for item in items]
    return batch


def make_loader(records, shuffle=False, batch_size=EVAL_BATCH_SIZE):
    return DataLoader(AspectDataset(records), batch_size=batch_size, shuffle=shuffle,
                      collate_fn=collate_batch, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))

train_loader = make_loader(train_records, shuffle=True, batch_size=BATCH_SIZE)
val_loader = make_loader(val_records)
test_loader = make_loader(test_records)
print(f"Từ vựng chỉ fit train: {len(char_vocab)} ký tự; {len(word_vocab)} từ.")
print(f"Độ dài ký tự lớn nhất trong train: {max(len(r['text']) for r in train_records)}; không truncation.")


## 5. Kiến trúc mô hình
BiLSTM hai lớp đọc hai chiều trên embedding ký tự kết hợp từ. CRF học chuyển trạng thái BIO và giải mã Viterbi hợp lệ. Polarity dùng đầu/cuối, trung bình span và trung bình câu; huấn luyện trên gold span, suy luận trên span dự đoán.


In [ ]:
class LinearChainCRF(nn.Module):
    """CRF O/B/I tự cài đặt, chặn I ở đầu câu và chuyển O→I."""
    def __init__(self):
        super().__init__()
        self.transitions = nn.Parameter(torch.zeros(3, 3))  # trước, sau
        self.start = nn.Parameter(torch.zeros(3))
        self.end = nn.Parameter(torch.zeros(3))
        legal = torch.ones(3, 3, dtype=torch.bool)
        legal[0, 2] = False
        self.register_buffer("legal", legal)
        self.register_buffer("legal_start", torch.tensor([True, True, False]))

    def constrained(self):
        return (self.transitions.masked_fill(~self.legal, -10000.0),
                self.start.masked_fill(~self.legal_start, -10000.0))

    def neg_log_likelihood(self, emissions, tags, mask):
        transitions, start = self.constrained()
        alpha = start + emissions[:, 0]
        rows = torch.arange(tags.size(0), device=tags.device)
        gold = start[tags[:, 0]] + emissions[rows, 0, tags[:, 0]]
        for t in range(1, emissions.size(1)):
            scores = alpha[:, :, None] + transitions[None] + emissions[:, t, None, :]
            updated = torch.logsumexp(scores, dim=1)
            alpha = torch.where(mask[:, t, None], updated, alpha)
            step = transitions[tags[:, t - 1], tags[:, t]] + emissions[rows, t, tags[:, t]]
            gold = gold + step * mask[:, t]
        lengths = mask.sum(1)
        last = tags.gather(1, (lengths - 1)[:, None]).squeeze(1)
        gold = gold + self.end[last]
        partition = torch.logsumexp(alpha + self.end, dim=1)
        return ((partition - gold) / lengths).mean()

    @torch.no_grad()
    def decode(self, emissions, mask):
        transitions, start = self.constrained()
        score = start + emissions[:, 0]
        history = []
        for t in range(1, emissions.size(1)):
            next_score, back = (score[:, :, None] + transitions[None]).max(dim=1)
            next_score = next_score + emissions[:, t]
            score = torch.where(mask[:, t, None], next_score, score)
            history.append(back)
        last = (score + self.end).argmax(dim=1)
        paths = []
        for i, length in enumerate(mask.sum(1).tolist()):
            tag = int(last[i])
            path = [tag]
            for t in range(length - 2, -1, -1):
                tag = int(history[t][i, tag])
                path.append(tag)
            paths.append(path[::-1])
        return paths


In [ ]:
class AspectModel(nn.Module):
    def __init__(self, cfg, n_chars, n_words):
        super().__init__()
        self.char_embedding = nn.Embedding(n_chars, cfg["char_dim"], padding_idx=PAD_ID)
        self.word_embedding = nn.Embedding(n_words, cfg["word_dim"], padding_idx=PAD_ID)
        self.dropout = nn.Dropout(cfg["dropout"])
        self.encoder = nn.LSTM(cfg["char_dim"] + cfg["word_dim"], cfg["hidden_dim"],
                               num_layers=cfg["num_layers"], batch_first=True,
                               bidirectional=True, dropout=cfg["dropout"])
        dim = 2 * cfg["hidden_dim"]
        self.tag_head = nn.Linear(dim, 3)
        self.crf = LinearChainCRF()
        self.polarity_head = nn.Sequential(nn.Linear(4 * dim, cfg["hidden_dim"]),
                                          nn.ReLU(), nn.Dropout(cfg["dropout"]),
                                          nn.Linear(cfg["hidden_dim"], len(LABELS)))

    def forward(self, batch):
        embedded = torch.cat([self.char_embedding(batch["char_ids"]),
                              self.word_embedding(batch["word_ids"])], dim=-1)
        packed = pack_padded_sequence(self.dropout(embedded), batch["lengths"],
                                       batch_first=True, enforce_sorted=False)
        packed_hidden, _ = self.encoder(packed)
        hidden, _ = pad_packed_sequence(packed_hidden, batch_first=True,
                                         total_length=batch["char_ids"].size(1))
        hidden = self.dropout(hidden)
        return hidden, self.tag_head(hidden)

    def extraction_loss(self, emissions, batch):
        return self.crf.neg_log_likelihood(emissions, batch["tags"], batch["mask"])

    def decode(self, emissions, mask):
        return self.crf.decode(emissions, mask)

    def classify_spans(self, hidden, spans, mask):
        if spans.numel() == 0:
            return hidden.new_empty((0, len(LABELS)))
        row, start, end = spans[:, 0], spans[:, 1], spans[:, 2]
        prefix = torch.cat([hidden.new_zeros(hidden.size(0), 1, hidden.size(2)),
                            hidden.cumsum(dim=1)], dim=1)
        mean = (prefix[row, end] - prefix[row, start]) / (end - start).unsqueeze(1)
        sentence = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)
        features = torch.cat([hidden[row, start], hidden[row, end - 1], mean, sentence[row]], -1)
        return self.polarity_head(features)


In [ ]:
MODEL_CONFIG = {"char_dim": CHAR_EMBED_DIM, "word_dim": WORD_EMBED_DIM,
                "hidden_dim": HIDDEN_DIM, "num_layers": NUM_LAYERS, "dropout": DROPOUT}
model = AspectModel(MODEL_CONFIG, len(char_vocab), len(word_vocab)).to(DEVICE)

print(f"Số tham số học từ đầu: {sum(p.numel() for p in model.parameters()):,}")


## 6. Metric và đánh giá
Aspect span micro-F1 khớp chính xác `(sid, start, end)` trên mọi câu; end loại trừ, không trim. Polarity macro-F1 tính trên **gold span**, cố định bốn lớp, lớp không có support được F1=0. Báo cáo thêm micro-F1/accuracy đúng ngữ cảnh SB2 và joint span+polarity F1 cho toàn pipeline. Không chỉ chấm polarity trên những span trích xuất đúng. Metric thiếu nhãn được trả `None`, không giả thành 0.


In [ ]:
def move_batch(batch):
    return {name: (value.to(DEVICE) if torch.is_tensor(value) and name != "lengths" else value)
            for name, value in batch.items()}


def bio_to_spans(tags):
    # I sau O được sửa thành B; cùng quy tắc ở cả hai biến thể. Không trim khoảng trắng.
    spans, start = [], None
    for index, tag in enumerate(list(tags) + [0]):
        if tag in (0, 1) and start is not None:
            spans.append((start, index))
            start = None
        if tag == 1 or (tag == 2 and start is None):
            start = index
    return spans


def set_prf(gold, predicted):
    tp = len(gold & predicted)
    precision = tp / len(predicted) if predicted else 0.0
    recall = tp / len(gold) if gold else 0.0
    f1 = 2 * tp / (len(gold) + len(predicted)) if gold or predicted else 0.0
    return {"precision": precision, "recall": recall, "f1": f1,
            "tp": tp, "predicted": len(predicted), "gold": len(gold)}


def classification_metrics(gold, predicted):
    if not gold:
        return {"macro_f1": None, "micro_f1": None, "accuracy": None, "count": 0,
                "per_class": None, "confusion_matrix": None}
    confusion = np.zeros((len(LABELS), len(LABELS)), dtype=np.int64)
    for truth, guess in zip(gold, predicted):
        confusion[truth, guess] += 1
    per_class = {}
    for i, label in enumerate(LABELS):
        tp = int(confusion[i, i])
        support = int(confusion[i].sum())
        predicted_count = int(confusion[:, i].sum())
        denominator = support + predicted_count
        per_class[label] = {"f1": 2 * tp / denominator if denominator else 0.0,
                            "support": support}
    accuracy = float(np.trace(confusion) / confusion.sum())
    return {"macro_f1": float(np.mean([v["f1"] for v in per_class.values()])),
            "micro_f1": accuracy, "accuracy": accuracy, "count": len(gold),
            "per_class": per_class, "confusion_matrix": confusion.tolist()}


@torch.no_grad()
def evaluate(model, loader, include_predictions=False):
    model.eval()
    gold_aspects, predicted_aspects, gold_joint, predicted_joint = set(), set(), set(), set()
    polarity_gold, polarity_predicted, examples = [], [], []
    joint_sentence_count = 0
    for cpu_batch in loader:
        batch = move_batch(cpu_batch)
        hidden, emissions = model(batch)
        paths = model.decode(emissions, batch["mask"])
        predicted_spans = [bio_to_spans(path) for path in paths]
        candidates = [(i, a, b) for i, spans in enumerate(predicted_spans) for a, b in spans]
        span_tensor = torch.tensor(candidates, dtype=torch.long, device=DEVICE).reshape(-1, 3)
        logits = model.classify_spans(hidden, span_tensor, batch["mask"])
        guesses = logits.argmax(-1).tolist() if len(candidates) else []
        by_sentence = defaultdict(list)
        for (i, a, b), label in zip(candidates, guesses):
            by_sentence[i].append((a, b, LABELS[label]))
        gold_span_tensor = batch["gold_spans"]
        gold_logits = model.classify_spans(hidden, gold_span_tensor[:, :3], batch["mask"])
        if len(gold_span_tensor):
            polarity_gold.extend(gold_span_tensor[:, 3].tolist())
            polarity_predicted.extend(gold_logits.argmax(-1).tolist())
        for i, record in enumerate(batch["records"]):
            sid = record["sid"]
            gold_aspects.update((sid, a, b) for a, b, _ in record["spans"])
            predicted_aspects.update((sid, a, b) for a, b, _ in by_sentence[i])
            if all(p is not None for _, _, p in record["spans"]):
                joint_sentence_count += 1
                gold_joint.update((sid, a, b, p) for a, b, p in record["spans"])
                predicted_joint.update((sid, a, b, p) for a, b, p in by_sentence[i])
            if include_predictions:
                examples.append({"text": record["text"], "aspects": [
                    {"start": a, "end": b, "span": record["text"][a:b], "polarity": p}
                    for a, b, p in by_sentence[i]]})
    scores = {"aspect_span": set_prf(gold_aspects, predicted_aspects),
              "polarity_gold_span": classification_metrics(polarity_gold, polarity_predicted),
              "joint_span_polarity": set_prf(gold_joint, predicted_joint) if joint_sentence_count else None,
              "joint_labeled_sentence_count": joint_sentence_count}
    return (scores, examples) if include_predictions else scores


## 7. Huấn luyện và lưu checkpoint
Loss extraction cộng 0.25 × CE polarity trên gold span. Chọn checkpoint duy nhất theo validation aspect F1; hòa điểm giữ epoch sớm hơn. Early stopping patience=8; test chỉ được đánh giá sau khi tải lại checkpoint tốt nhất.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
best_f1, stale_epochs = -1.0, 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    totals, sentence_count = np.zeros(3), 0
    for cpu_batch in train_loader:
        batch = move_batch(cpu_batch)
        optimizer.zero_grad(set_to_none=True)
        hidden, emissions = model(batch)
        extraction_loss = model.extraction_loss(emissions, batch)
        spans = batch["gold_spans"]
        if len(spans):
            polarity_logits = model.classify_spans(hidden, spans[:, :3], batch["mask"])
            polarity_loss = nn.functional.cross_entropy(polarity_logits, spans[:, 3])
        else:
            polarity_loss = hidden.sum() * 0.0
        loss = extraction_loss + POLARITY_LOSS_WEIGHT * polarity_loss
        if not torch.isfinite(loss):
            raise FloatingPointError("Loss không hữu hạn; kiểm tra dữ liệu và cấu hình cố định.")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        count = len(batch["records"])
        totals += np.array([loss.item(), extraction_loss.item(), polarity_loss.item()]) * count
        sentence_count += count
    losses = dict(zip(("loss", "extraction_loss", "polarity_loss"), (totals / sentence_count).tolist()))
    metrics = evaluate(model, val_loader)
    score = metrics["aspect_span"]["f1"]
    polarity_f1 = metrics["polarity_gold_span"]["macro_f1"]
    print(f"Epoch {epoch:02d}: loss={losses['loss']:.4f}; "
          f"validation aspect F1={score:.4f}; polarity macro-F1={polarity_f1:.4f}")
    if score > best_f1:  # Hòa điểm: giữ checkpoint sớm hơn, không nhìn test.
        best_f1, stale_epochs = score, 0
        torch.save({
            "model_state": model.state_dict(),
            "char_vocab": char_vocab,
            "word_vocab": word_vocab,
            "labels": list(LABELS),
            "tags": list(TAGS),
            "epoch": epoch,
        }, CHECKPOINT_PATH)
        print("Đã lưu checkpoint tốt nhất theo aspect span F1 trên validation.")
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f"Dừng sớm sau {PATIENCE} epoch không cải thiện.")
            break


## 8. Tải lại và đánh giá cuối
Checkpoint lưu trọng số, optimizer, kiến trúc, vocabulary, nhãn, decoder, seed, phiên bản và fingerprint split. Khởi tạo lại mô hình từ checkpoint trước khi chấm. Test không có polarity gold: vẫn xuất nhãn dự đoán nhưng không báo điểm polarity test. `metrics.json` và `test_predictions.json` được tạo khi người dùng chạy notebook.


In [ ]:
# Chỉ tải checkpoint do chính notebook này tạo.
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
assert checkpoint["labels"] == list(LABELS) and checkpoint["tags"] == list(TAGS)
char_vocab, word_vocab = checkpoint["char_vocab"], checkpoint["word_vocab"]
# Cấu hình kiến trúc ở notebook; kích thước vocab được lấy sau khi restore.
inference_config = {"char_dim": CHAR_EMBED_DIM, "word_dim": WORD_EMBED_DIM,
                "hidden_dim": HIDDEN_DIM, "num_layers": NUM_LAYERS, "dropout": DROPOUT}
best_model = AspectModel(inference_config, len(char_vocab), len(word_vocab)).to(DEVICE)
best_model.load_state_dict(checkpoint["model_state"])
best_model.eval()
# Tạo lại loader từ vocabulary đã tải; không phụ thuộc state của mô hình cũ.
final_val_metrics = evaluate(best_model, make_loader(val_records))
test_metrics, test_predictions = evaluate(best_model, make_loader(test_records), include_predictions=True)
print(f"Đã nạp checkpoint epoch {checkpoint['epoch']}; mọi metric ở thang 0–1.")
print("Validation:", json.dumps(final_val_metrics, ensure_ascii=False, indent=2))
print("Test:", json.dumps(test_metrics, ensure_ascii=False, indent=2))
if test_metrics["polarity_gold_span"]["count"] == 0:
    print("Test không có nhãn polarity: không tính polarity F1/accuracy hay joint F1 trên test.")
results = {"variant": VARIANT, "epoch": checkpoint["epoch"],
           "dataset_revision": DATASET_REVISION, "split_fingerprints": SPLIT_FINGERPRINTS,
           "validation": final_val_metrics, "test": test_metrics}
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "test_predictions.json").write_text(
    json.dumps(test_predictions, ensure_ascii=False, indent=2), encoding="utf-8")
print("Dự đoán test đầu tiên:", test_predictions[0])


## 9. Dự đoán câu mới
Hàm trả ranh giới ký tự và polarity của từng aspect; có thể trả danh sách rỗng. Đây là suy luận, không phải minh chứng điểm chất lượng.


In [ ]:
@torch.no_grad()
def predict_text(text):
    if not isinstance(text, str) or not text:
        raise ValueError("Cần một câu không rỗng.")
    record = {"sid": hashlib.sha256(text.encode()).hexdigest(), "text": text, "spans": []}
    batch = move_batch(collate_batch([encode_record(record)]))
    best_model.eval()
    hidden, emissions = best_model(batch)
    spans = bio_to_spans(best_model.decode(emissions, batch["mask"])[0])
    candidates = torch.tensor([(0, a, b) for a, b in spans], device=DEVICE, dtype=torch.long).reshape(-1, 3)
    logits = best_model.classify_spans(hidden, candidates, batch["mask"])
    polarities = logits.argmax(-1).tolist() if spans else []
    return [{"start": a, "end": b, "span": text[a:b], "polarity": LABELS[p]}
            for (a, b), p in zip(spans, polarities)]

print(predict_text("The food is great but the service is slow."))
